In [ ]:
import mlflow
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [ ]:
df_train = pd.read_csv("./data/train.csv")
df_test = pd.read_csv("./data/test.csv")
df_train.head()

In [ ]:
# 2. 데이터 준비
X_train = df_train.drop(["type", "product_id", "target"], axis=1)
y_train = df_train["target"]

X_test = df_test.drop(["type", "product_id", "target"], axis=1)
y_test = df_test["target"]

mlflow.set_tracking_uri("http://127.0.0.1:5000")
exp = mlflow.set_experiment(experiment_name="기본 모델 학습")
with mlflow.start_run(experiment_id=exp.experiment_id):  # , log_system_metrics=True
    # 3. 모델 생성 및 학습
    model = LogisticRegression(max_iter=1000)
    # 4. 모델 학습
    print("모델 학습을 시작합니다...")
    model.fit(X_train, y_train)
    print("모델 학습이 완료되었습니다.")

    # 파라미터 로깅
    mlflow.log_param("max_iter", 1000)

    # 5. 예측
    # 학습된 모델을 사용하여 테스트 데이터의 결과를 예측
    y_pred = model.predict(X_test)

    # 6. 모델 평가
    # 실제 값(y_test)과 예측 값(y_pred)을 비교하여 모델 성능 평가
    accuracy = accuracy_score(y_test, y_pred)

    # 정확도 로깅
    print(f"\n모델 정확도(Accuracy): {accuracy:.4f}")

    mlflow.log_metric("test_accuracy", accuracy)
    mlflow.sklearn.log_model(sk_model=model, name="logistic_regression")

    mlflow.log_table(df_train.head(), artifact_file="train.json")
    mlflow.log_table(df_test.head(), artifact_file="test.json")
    # artifact
    mlflow.log_artifact("./data/train.csv", artifact_path="data")
    filename = "my_first_model.joblib"
    joblib.dump(model, filename)
    mlflow.log_artifact(filename, artifact_path="model")
